# Лабораторная работа № 2 — ЭТАЛОННОЕ РЕШЕНИЕ

## Группа вращений SO(3): экспоненциальное отображение, кватернионы, интерполяция

**Дисциплина:** Введение в технологии виртуальной дополненной реальности и робототехники
**Тема курса:** 3. Группа вращений SO(3): матрицы, углы Эйлера, экспоненциальное отображение, кватернионы

---

**Для преподавателя.** Не выдавать студентам до сдачи работы.

---

### Цель работы

Освоить три способа представления ориентации — матрицу, вектор поворота и кватернион, —
реализовать переходы между ними, обнаружить численно вырождение углов Эйлера
и сравнить представления по устойчивости при длительном интегрировании.

### Что понадобится

`numpy`, `matplotlib`. Из лабораторной работы № 1 переиспользуются `rotx`, `roty`, `rotz`.

### Правило практикума

Алгоритм вы реализуете сами. `scipy.spatial.transform.Rotation` допустимо использовать
**только** для сверки — и такую сверку нужно показать явно.

## Теоретическая справка

### Экспоненциальное отображение

Всякий поворот есть поворот вокруг некоторой оси (теорема Эйлера). Если $\hat\omega$ —
единичный вектор оси, $\theta$ — угол, то поворот задаётся **вектором поворота**
$\omega = \theta\hat\omega \in \mathbb{R}^3$, а матрица получается по формуле Родрига:

$$R = \exp([\omega]_\times) = I + \sin\theta\,[\hat\omega]_\times
  + (1-\cos\theta)\,[\hat\omega]_\times^2,$$

где $[\cdot]_\times$ — кососимметрическая матрица векторного произведения. Обратное
отображение:

$$\theta = \arccos\frac{\operatorname{tr}R - 1}{2},
\qquad
[\hat\omega]_\times = \frac{R - R^{\mathsf T}}{2\sin\theta}.$$

Множество кососимметрических матриц $3\times3$ — это алгебра Ли $\mathfrak{so}(3)$,
касательное пространство к $SO(3)$ в единице. Она трёхмерна, и это ещё одно объяснение
того, почему у поворота три параметра.

**Особые случаи, которые придётся обработать явно:** $\theta \to 0$ (деление на
$\sin\theta$) и $\theta \to \pi$ (матрица $R - R^{\mathsf T}$ обращается в нуль, ось
приходится извлекать из $R + I$).

### Кватернионы

Единичный кватернион $q = (w, x, y, z)$, $\|q\| = 1$, задаёт поворот на угол $\theta$
вокруг оси $\hat\omega$ как

$$q = \left(\cos\tfrac{\theta}{2},\; \hat\omega \sin\tfrac{\theta}{2}\right).$$

Композиция поворотов — произведение кватернионов; обращение — сопряжение.
Отображение $q \mapsto R(q)$ является **двойным накрытием**: $q$ и $-q$ задают один
и тот же поворот. Отсюда практическое следствие для интерполяции: перед началом нужно
выбрать знак второго кватерниона так, чтобы скалярное произведение было неотрицательным,
иначе интерполяция пойдёт «длинным путём».

### Сферическая линейная интерполяция

$$\mathrm{slerp}(q_0, q_1, t) =
\frac{\sin((1-t)\Omega)\,q_0 + \sin(t\Omega)\,q_1}{\sin\Omega},
\qquad \cos\Omega = q_0 \cdot q_1 .$$

Это движение по дуге большого круга на сфере $S^3$ с постоянной угловой скоростью.
Покомпонентная интерполяция углов Эйлера такого свойства не даёт — в этом вы убедитесь
численно в задании 5.

### Вырождение углов Эйлера

Связь производных углов с угловой скоростью для последовательности ZYX имеет вид
$\omega = E(\psi,\theta,\varphi)\,(\dot\psi, \dot\theta, \dot\varphi)^{\mathsf T}$, причём
$\det E = -\cos\theta$. При $\theta \to \pm 90^\circ$ матрица вырождается: две оси
поворота совпадают, теряется степень свободы, а обратная задача становится
неограниченно плохо обусловленной. Это и есть **gimbal lock**.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)


def rotx(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def roty(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def rotz(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

## Задание 1. Экспоненциальное и логарифмическое отображения

Ожидаемо: максимум ошибки по 300 матрицам порядка $10^{-14}$. Без обработки особого случая $\theta \approx \pi$ проверка падает примерно на одной матрице из тысячи.

In [2]:
def skew(w):
    wx, wy, wz = w
    return np.array([[0,-wz,wy],[wz,0,-wx],[-wy,wx,0]])

def unskew(K):
    return np.array([K[2,1], K[0,2], K[1,0]])

def so3_exp(w):
    th = np.linalg.norm(w)
    if th < 1e-12:
        return np.eye(3) + skew(w)
    K = skew(w / th)
    return np.eye(3) + np.sin(th)*K + (1-np.cos(th))*(K @ K)

def so3_log(R):
    c = np.clip((np.trace(R) - 1) / 2, -1.0, 1.0)
    th = np.arccos(c)
    if th < 1e-8:
        return unskew(R - R.T) / 2
    if np.pi - th < 1e-6:                       # θ близко к π
        A = (R + np.eye(3)) / 2
        ax = np.sqrt(np.maximum(np.diag(A), 0))
        k = int(np.argmax(ax)); ax = ax / ax[k] * ax[k]
        ax = ax * np.sign(A[k] / (ax[k] if ax[k] > 0 else 1))
        ax = ax / np.linalg.norm(ax)
        return th * ax
    return th / (2*np.sin(th)) * unskew(R - R.T)

def rand_R(rng):
    w = rng.normal(size=3); w = w / np.linalg.norm(w) * rng.uniform(0.05, 3.0)
    return so3_exp(w)

err = max(np.linalg.norm(so3_exp(so3_log(R := rand_R(rng))) - R) for _ in range(300))
print('Задание 1: max ||exp(log R) − R|| по 300 матрицам = %.2e' % err)
assert err < 1e-9
w0 = np.array([0.3, -1.1, 0.7])
print('           ||log(exp w) − w|| = %.2e' % np.linalg.norm(so3_log(so3_exp(w0)) - w0))

Задание 1: max ||exp(log R) − R|| по 300 матрицам = 7.91e-14
           ||log(exp w) − w|| = 2.72e-16


## Задание 2. Переходы между представлениями

Ожидаемо: кватернионы дают $\sim10^{-15}$, углы Эйлера $\sim10^{-15}\ldots10^{-14}$. **Ответ на вопрос:** проход через углы Эйлера включает `arcsin`, производная которого неограниченно растёт у краёв области определения, поэтому вблизи тангажа $\pm90^\circ$ точность падает. Кватернионная ветвь состоит только из умножений и одного корня.

In [3]:
def R_to_quat(R):
    tr = np.trace(R)
    if tr > 0:
        s = np.sqrt(tr + 1.0) * 2
        q = np.array([0.25*s, (R[2,1]-R[1,2])/s, (R[0,2]-R[2,0])/s, (R[1,0]-R[0,1])/s])
    else:
        i = int(np.argmax(np.diag(R))); j, k = (i+1) % 3, (i+2) % 3
        s = np.sqrt(1.0 + R[i,i] - R[j,j] - R[k,k]) * 2
        q = np.empty(4)
        q[0] = (R[k,j] - R[j,k]) / s
        q[1+i] = 0.25 * s
        q[1+j] = (R[j,i] + R[i,j]) / s
        q[1+k] = (R[k,i] + R[i,k]) / s
    return q / np.linalg.norm(q)

def quat_to_R(q):
    w, x, y, z = q / np.linalg.norm(q)
    return np.array([
        [1-2*(y*y+z*z), 2*(x*y-w*z),   2*(x*z+w*y)],
        [2*(x*y+w*z),   1-2*(x*x+z*z), 2*(y*z-w*x)],
        [2*(x*z-w*y),   2*(y*z+w*x),   1-2*(x*x+y*y)]])

def R_to_euler_zyx(R):
    pitch = np.arcsin(np.clip(-R[2,0], -1, 1))
    if abs(R[2,0]) < 1 - 1e-9:
        return np.array([np.arctan2(R[1,0], R[0,0]), pitch, np.arctan2(R[2,1], R[2,2])])
    return np.array([np.arctan2(-R[0,1], R[1,1]), pitch, 0.0])   # вырождение

def euler_zyx_to_R(a):
    return rotz(a[0]) @ roty(a[1]) @ rotx(a[2])

e1 = e2 = 0.0
for _ in range(300):
    R = rand_R(rng)
    e1 = max(e1, np.linalg.norm(quat_to_R(R_to_quat(R)) - R))
    e2 = max(e2, np.linalg.norm(euler_zyx_to_R(R_to_euler_zyx(R)) - R))
print('Задание 2: матрица→кватернион→матрица  max err = %.2e' % e1)
print('           матрица→углы Эйлера→матрица max err = %.2e' % e2)
assert e1 < 1e-12 and e2 < 1e-10

Задание 2: матрица→кватернион→матрица  max err = 1.15e-15
           матрица→углы Эйлера→матрица max err = 4.19e-15


## Задание 3. Двойное накрытие

**Ответ на вопрос:** при интерполяции знак определяет, пойдёт ли движение коротким путём или длинным (через угол больше $180^\circ$); снимается проверкой знака скалярного произведения. При усреднении набора ориентаций знаки надо согласовать заранее, иначе кватернионы взаимно уничтожатся.

In [4]:
R = rand_R(rng); q = R_to_quat(R)
print('Задание 3: ||R(q) − R(−q)|| = %.2e  (q и −q задают один поворот)'
      % np.linalg.norm(quat_to_R(q) - quat_to_R(-q)))
assert np.linalg.norm(quat_to_R(q) - quat_to_R(-q)) < 1e-14

Задание 3: ||R(q) − R(−q)|| = 0.00e+00  (q и −q задают один поворот)


## Задание 4. Gimbal lock — численное обнаружение

**Ответ на вопросы:** $|\det E| = |\cos\theta|$ — численный результат совпадает с аналитическим до машинной точности. Число обусловленности растёт обратно пропорционально расстоянию до вырождения: уменьшение угла в десять раз увеличивает его примерно в десять раз ($114 \to 1146 \to 11459$). Механически в этот момент две оси карданова подвеса становятся параллельными и одна степень свободы теряется.

In [5]:
def euler_rate_matrix(a):
    """ω = E(a) · (ψ̇, θ̇, φ̇) для последовательности ZYX."""
    psi, th, _ = a
    return np.column_stack([np.array([0,0,1.0]),
                            rotz(psi) @ np.array([0,1.0,0]),
                            rotz(psi) @ roty(th) @ np.array([1.0,0,0])])

pitches = np.deg2rad(np.array([0, 30, 60, 80, 89, 89.9, 89.99]))
conds = [np.linalg.cond(euler_rate_matrix([0.4, p, 0.2])) for p in pitches]
dets = [abs(np.linalg.det(euler_rate_matrix([0.4, p, 0.2]))) for p in pitches]
print('Задание 4: тангаж (град) ->', np.round(np.rad2deg(pitches), 2))
print('           число обусл.  ->', np.array2string(np.array(conds), precision=1))
print('           |det E| = |cos θ| ->', np.array2string(np.array(dets), precision=5))
assert conds[-1] > 1e3 and dets[-1] < 1e-3

Задание 4: тангаж (град) -> [ 0.   30.   60.   80.   89.   89.9  89.99]
           число обусл.  -> [    1.      1.7     3.7    11.4   114.6  1145.9 11459.2]
           |det E| = |cos θ| -> [1.      0.86603 0.5     0.17365 0.01745 0.00175 0.00017]


## Задание 5. SLERP против интерполяции углов Эйлера

**Ответ на вопросы:** у SLERP разброс $\sim10^{-13}$, то есть скорость постоянна с точностью до машинной погрешности; у интерполяции углов Эйлера разброс порядка единицы — почти треть от среднего значения. В VR это критично при асинхронном репроецировании: кадр доворачивается на разницу поз, и неравномерность интерполяции воспринимается как рывок.

In [6]:
def slerp(q0, q1, t):
    q0 = q0/np.linalg.norm(q0); q1 = q1/np.linalg.norm(q1)
    d = np.dot(q0, q1)
    if d < 0:
        q1, d = -q1, -d                       # кратчайший путь
    if d > 0.9995:
        return (q0 + t*(q1-q0)) / np.linalg.norm(q0 + t*(q1-q0))
    Om = np.arccos(np.clip(d, -1, 1))
    return (np.sin((1-t)*Om)*q0 + np.sin(t*Om)*q1) / np.sin(Om)

R0 = euler_zyx_to_R([0.0, 0.0, 0.0])
R1 = euler_zyx_to_R([2.2, -1.0, 1.7])
q0, q1 = R_to_quat(R0), R_to_quat(R1)
a0, a1 = R_to_euler_zyx(R0), R_to_euler_zyx(R1)
ts = np.linspace(0, 1, 401)
Rs_s = [quat_to_R(slerp(q0, q1, t)) for t in ts]
Rs_e = [euler_zyx_to_R(a0 + t*(a1-a0)) for t in ts]

def ang_speed(Rs, ts):
    return np.array([np.linalg.norm(so3_log(Rs[i+1] @ Rs[i].T)) / (ts[i+1]-ts[i])
                     for i in range(len(Rs)-1)])

ws, we = ang_speed(Rs_s, ts), ang_speed(Rs_e, ts)
print('Задание 5: SLERP  — угловая скорость: среднее %.4f, разброс %.2e'
      % (ws.mean(), ws.max()-ws.min()))
print('           Эйлер — угловая скорость: среднее %.4f, разброс %.4f'
      % (we.mean(), we.max()-we.min()))
print('           у SLERP разброс на уровне машинной погрешности: скорость постоянна')
assert (ws.max()-ws.min()) < 1e-6 and (we.max()-we.min()) > 0.1

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(ts[:-1], ws, label='SLERP', lw=2)
ax.plot(ts[:-1], we, label='линейная интерполяция углов Эйлера', lw=2)
ax.set_xlabel('t'); ax.set_ylabel('|ω|'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig('/tmp/_slerp.png', dpi=80); plt.close()

Задание 5: SLERP  — угловая скорость: среднее 3.0250, разброс 3.69e-13
           Эйлер — угловая скорость: среднее 3.4779, разброс 0.9192
           у SLERP разброс на уровне машинной погрешности: скорость постоянна


## Задание 6. Дрейф при длительном интегрировании

**Ответ на вопросы:** матрица портится примерно на порядок сильнее — у неё девять чисел и шесть связей между ними, у кватерниона четыре числа и одна связь. Нормировка кватерниона — деление на норму, она возвращает точку строго на сферу; для матрицы аналогом была бы проекция на $SO(3)$, но она требует SVD. При частоте 1000 Гц $2\cdot10^5$ шагов — это чуть больше трёх минут работы.

In [7]:
w = np.array([0.7, -0.3, 1.1]); dt = 1e-3; N = 200000
dR = so3_exp(w*dt); dq = R_to_quat(dR)

def qmul(a, b):
    w1,x1,y1,z1 = a; w2,x2,y2,z2 = b
    return np.array([w1*w2-x1*x2-y1*y2-z1*z2, w1*x2+x1*w2+y1*z2-z1*y2,
                     w1*y2-x1*z2+y1*w2+z1*x2, w1*z2+x1*y2-y1*x2+z1*w2])

Rm = np.eye(3); qq = np.array([1.0,0,0,0]); qn = np.array([1.0,0,0,0])
step = N // 40
xs, dm, dqn, dqq = [], [], [], []
for i in range(1, N+1):
    Rm = Rm @ dR
    qq = qmul(qq, dq)
    qn = qmul(qn, dq); qn = qn / np.linalg.norm(qn)
    if i % step == 0:
        xs.append(i)
        dm.append(np.linalg.norm(Rm.T @ Rm - np.eye(3)))
        dqq.append(abs(np.linalg.norm(qq) - 1))
        dqn.append(abs(np.linalg.norm(qn) - 1))
print('Задание 6: после %d шагов' % N)
print('           матрица:    ||RᵀR − I||   = %.2e' % dm[-1])
print('           кватернион: | ||q|| − 1 | = %.2e (без нормировки)' % dqq[-1])
print('           кватернион: | ||q|| − 1 | = %.2e (с нормировкой)' % dqn[-1])
print('           отношение матрица/кватернион = %.0f' % (dm[-1]/max(dqq[-1], 1e-18)))
assert dm[-1] > dqq[-1] and dqn[-1] < 1e-14

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.loglog(xs, dm, 'o-', label=r'матрица: $\|R^TR-I\|$')
ax.loglog(xs, np.maximum(dqq, 1e-18), 's-', label=r'кватернион: $|\,\|q\|-1|$')
ax.set_xlabel('число шагов'); ax.legend(); ax.grid(True, which='both', alpha=.3)
plt.tight_layout(); plt.savefig('/tmp/_drift.png', dpi=80); plt.close()

Задание 6: после 200000 шагов
           матрица:    ||RᵀR − I||   = 3.02e-11
           кватернион: | ||q|| − 1 | = 2.03e-12 (без нормировки)
           кватернион: | ||q|| − 1 | = 0.00e+00 (с нормировкой)
           отношение матрица/кватернион = 15


## Задание 7. Цена восстановления: SVD против нормировки

**Ответ на вопрос:** SVD примерно на порядок дороже нормировки. Вывод: там, где ориентация интегрируется на высокой частоте (инерциальный блок, трекинг шлема), её хранят кватернионом. Там, где ориентация участвует в геометрических выкладках и композициях с положением (кинематическая цепь, дерево кадров), удобнее матрица.

In [8]:
import time
Rb = rand_R(rng) + rng.normal(0, 1e-3, (3,3))
t0 = time.perf_counter()
for _ in range(20000):
    U, S, Vt = np.linalg.svd(Rb); _ = U @ np.diag([1,1,np.sign(np.linalg.det(U@Vt))]) @ Vt
t_svd = time.perf_counter() - t0
qb = R_to_quat(rand_R(rng)) + rng.normal(0, 1e-3, 4)
t0 = time.perf_counter()
for _ in range(20000):
    _ = qb / np.linalg.norm(qb)
t_q = time.perf_counter() - t0
print('Задание 7: SVD-ортогонализация %.1f мкс/шт, нормировка кватерниона %.2f мкс/шт '
      '(в %.0f раз быстрее)' % (t_svd/20000*1e6, t_q/20000*1e6, t_svd/t_q))
print('\nВСЕ ПРОВЕРКИ ПРОЙДЕНЫ')

Задание 7: SVD-ортогонализация 11.7 мкс/шт, нормировка кватерниона 1.04 мкс/шт (в 11 раз быстрее)

ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ


## Критерии приёма работы

| Что проверяется | На что смотреть |
|---|---|
| Задание 1 | Обработаны оба особых случая. Проверка на 300 матрицах, а не на одной |
| Задание 2 | `R_to_quat` реализован через выбор ветви по максимальному диагональному элементу |
| Задание 4 | Есть сравнение численного $|\det E|$ с аналитическим $|\cos\theta|$ |
| Задание 5 | Оба профиля на одном графике; названы конкретные значения разброса |
| Задание 6 | Три кривые, логарифмический масштаб, ответ про три минуты полёта |
| Выводы | Все «вопросы для вывода» разобраны с числами |
| Воспроизводимость | «Restart & Run All» отрабатывает без ошибок |

**Частые ошибки:**

1. `so3_log` без обработки $\theta \approx \pi$ — падает редко и потому не замечается.
   Требуйте проверку именно на большой случайной выборке.
2. Наивная формула `R_to_quat` через $\sqrt{1+\operatorname{tr}R}$ — теряет точность
   при больших углах.
3. В `slerp` забыт выбор знака — интерполяция иногда идёт «длинным путём».
   Проявляется не всегда, зависит от пары поворотов.
4. Угловая скорость считается как разность углов Эйлера, а не через $\log(R_{i+1}R_i^{\mathsf T})$.
   Тогда графики совпадают и весь смысл задания теряется.
5. В задании 6 берут слишком мало шагов: при $N = 10^3$ дрейф ещё не виден.

## Прикладной блок · SO(3) в ROS 2: кватернионы сообщений и exp/log в движках *(необязательный)*

Связка с ROS-стеком:

| Понятие из работы | Контрагент в ROS 2 |
|---|---|
| `so3_exp` / `so3_log` | `pin.exp3` / `pin.log3` — основа оптимизации на многообразиях в MoveIt и Pinocchio |
| `R_to_quat` (w, x, y, z) | `geometry_msgs/Quaternion` передаёт **x, y, z, w** — тот же кватернион в другом порядке полей |
| `slerp` | `Eigen::Quaternion::slerp` — интерполятор ориентаций в планировщиках |
| дрейф при интегрировании (задание 6) | поэтому в ROS-стеке ориентацию хранят кватернионом и нормируют |

В контейнере `ros2` окружения курса ориентацию видно «снаружи»:

```bash
ros2 topic pub -r 10 /joint_states sensor_msgs/msg/JointState '{position: [0.3, -0.2, 0.5]}'
ros2 topic echo /tf   # поле rotation: {x, y, z, w} — порядок из таблицы выше
```

Блок ниже сверяет наши реализации с библиотечными. Без Pinocchio — пропуск.

In [9]:
# Прикладной блок (необязательный): SO(3) в Pinocchio и ROS-сообщениях
try:
    import pinocchio as pin
except ImportError:
    pin = None

if pin is None:
    print('Pinocchio не установлен — блок пропущен.')
    print('Установка: pip install pin (или docker-окружение курса, docker/README.md).')
else:
    # 1) exp/log: pin.exp3 == so3_exp, pin.log3 == so3_log
    ws = rng.normal(size=(200, 3)) * rng.uniform(0.05, 3.0, (200, 1))
    e_exp = max(np.linalg.norm(pin.exp3(w) - so3_exp(w)) for w in ws)
    Rs = [rand_R(rng) for _ in range(200)]
    e_log = max(np.linalg.norm(pin.log3(R) - so3_log(R)) for R in Rs)
    print('exp: max ||pin.exp3(w) - so3_exp(w)|| = %.2e' % e_exp)
    print('log: max ||pin.log3(R) - so3_log(R)|| = %.2e' % e_log)
    assert e_exp < 1e-12 and e_log < 1e-12

    # 2) порядок компонент: наш (w,x,y,z) против ROS/Eigen (x,y,z,w)
    R = rand_R(rng)
    q_lab = R_to_quat(R)                 # порядок (w, x, y, z)
    c = pin.Quaternion(R).coeffs()       # порядок (x, y, z, w)
    lab_order = np.r_[c[3], c[0], c[1], c[2]]
    err_q = min(np.linalg.norm(lab_order - q_lab),
               np.linalg.norm(lab_order + q_lab))    # q и -q — один поворот
    print('кватернион из матрицы: расхождение (с учётом знака) = %.2e' % err_q)
    assert err_q < 1e-12
    print('в geometry_msgs/Quaternion поля: x=%.4f, y=%.4f, z=%.4f, w=%.4f'
          % tuple(c))

    # 3) интерполяция: ваш slerp и Eigen-Quaternion::slerp — одна кривая
    qp0, qp1 = pin.Quaternion(R0), pin.Quaternion(R1)
    ts_ch = np.linspace(0, 1, 21)
    err_slerp = 0.0
    for t in ts_ch[1:-1]:
        cc = qp0.slerp(float(t), qp1).coeffs()
        R_eigen = quat_to_R(np.r_[cc[3], cc[:3]])
        R_nash = quat_to_R(slerp(q0, q1, t))
        err_slerp = max(err_slerp, np.linalg.norm(R_eigen - R_nash))
    print('slerp: max || Eigen(R) - наша(R) || в %d точках = %.2e'
          % (len(ts_ch) - 2, err_slerp))
    assert err_slerp < 1e-12
    print()
    print('ПРИКЛАДНОЙ БЛОК: ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ')

exp: max ||pin.exp3(w) - so3_exp(w)|| = 1.27e-15
log: max ||pin.log3(R) - so3_log(R)|| = 4.71e-14
кватернион из матрицы: расхождение (с учётом знака) = 1.21e-16
в geometry_msgs/Quaternion поля: x=-0.1636, y=0.1947, z=0.1949, w=0.9473
slerp: max || Eigen(R) - наша(R) || в 19 точках = 7.28e-16

ПРИКЛАДНОЙ БЛОК: ВСЕ ПРОВЕРКИ ПРОЙДЕНЫ
